# 01b - LLM Performance Analysis with GenAI Perf tool


# Introduction<a name="setup-inference-testing"></a>

The goal of this notebook is to show how to use the [genai-perf tool](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/perf_benchmark/genai_perf.html) that is part of the triton sdk package.<br>


There three major items that consume GPU memory for LLM inference: 
* model weights
* prefill creation of the [KV Cache](https://www.youtube.com/watch?v=Mn_9W1nCFLo)
* generation (where tokens are generated after the context is loaded into memory). 
  
Read more in depth [here](https://developer.nvidia.com/blog/mastering-llm-techniques-inference-optimization/). <br>

LLM Inference Optimization can involve optimizing the following factors:  
- Hardware used (memory, compute cores) - GPUs that have [FP8 support](https://arxiv.org/abs/2209.05433) (Hopper + Ada Lovelace) vs others that don't 
- Precision of Weights and Activations
    - [SmoothQuant: Accurate and Efficient Post-Training Quantization for Large Language Models](https://arxiv.org/abs/2209.05433)
    - [GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers](https://arxiv.org/abs/2210.1732)
    - AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration](https://arxiv.org/abs/2306.0098) 
    - [Tensor and Pipeline Parallelism](https://arxiv.org/pdf/2104.04473.pdf)  
    - [Types of decoding](https://towardsdatascience.com/the-three-decoding-methods-for-nlp-23ca59cb1e9d)  
    - Types of positional encoding - [RoFormer: Enhanced Transformer with Rotary Position Embedding]https://arxiv.org/abs/2104.09864(  
    - Inference Engine Build Parameters (in TensorRT-LLM) such as batch_size, max_output_token_length and others  

In order to tune all these parameters effectively, it is important to understand the distribution of traffic coming into your LLM service and the lengths and number of generations coming out of your service.  

  
**Notebook Requirements**
- Access to single NVIDIA GPU (A30, A100, H100, L40S)
- Access to [NGC](ngc.nvidia.com) 
- Docker

In [ ]:
# Check which GPU you have and how much memory we have to work with.
!nvidia-smi

## Setup for Inference Testing <a name="setup-inference-testing"></a>

Gen AI Perf Example 

In [ ]:
# If Nim Server is not running, start it up cleanly
# !bash scripts/stop_containers.sh
# !bash scripts/run_llama_nim.sh

In [ ]:
# Setup Environment

import os
import json

from dotenv import load_dotenv
# Load the .env file
load_dotenv('./docker/.env', override=True)

print(f"Current working dir           : { os.getcwd() }")
print(f"locally hosted model for lab  : {os.getenv('NIM_SERVED_MODEL_NAME')}")
print(f"NGC_API_KEY for lab use only  : {os.getenv('NGC_API_KEY')}")

data = {
    "model": os.getenv("NIM_SERVED_MODEL_NAME"),
    "messages": [{"role": "user", "content": "What are the main branches of the DoD?"}],
    "max_tokens": 100,
    "temperature": 0.5,
    "stream": False
}

cmd = f'curl -X POST "http://0.0.0.0:8000/v1/chat/completions" -H "accept: application/json" -H "Content-Type: application/json" -d \'{json.dumps(data)}\''

print(cmd)
!{cmd}|jq

## Example of running genai-perf manually

### 1) Start Docker Container
```bash
docker run -it --net=host --gpus=all  nvcr.io/nvidia/tritonserver:24.03-py3-sdk
```
### 2) Inside container run a genai-perf job.  Eg
```bash
genai-perf -m mistral-7b-chat
  --input-type synthetic 
  --output-format openai_chat_completions 
  --num-of-output-prompts 100 
  --input-tokens-mean 1000 
  --input-tokens-stddev 5 
  --expected-output-tokens 100 
  --random-seed 1   
  --concurrency 20 
  --service-kind openai 
  --endpoint v1/chat/completions 
  --streaming --u 0.0.0.0:9999  
  -- --max-threads 256
```

Click this link to see an example script we can use to test out our NIM running within Launchpad. <br>
[run genai-perf script](./scripts/run_genaiperf.sh)

In [ ]:
# Run the genai-perf command.  Ouputs results to a table.  Inspect documentation for running other modes.
!mkdir -p logs
!docker run --rm -it \
    -v ./:/nim-workshop \
    --net=host \
    --gpus '"device=0"'  \
    nvcr.io/nvidia/tritonserver:24.12-py3-sdk  \
    /nim-workshop/scripts/run_genaiperf.sh


In [ ]:
# Inspect /tmp/results.csv
!tree logs
!cat logs/profile_export_genai_perf.csv
    